In [31]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor
)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [32]:
DATA_PATH = Path("../data/processed")

train_data = pd.read_csv(DATA_PATH / "train_data.csv")
valid_data = pd.read_csv(DATA_PATH / "valid_data.csv")

train_data["date"] = pd.to_datetime(train_data["date"])
valid_data["date"] = pd.to_datetime(valid_data["date"])

KeyboardInterrupt: 

In [ ]:
SAMPLE_SIZE = 200_000

train_sample = train_data.sample(
    n=SAMPLE_SIZE,
    random_state=42
)

In [ ]:
print(train_sample.shape)
print(valid_data.shape)

(200000, 42)
(167508, 42)


In [ ]:
TARGET = "sales"

X_train = train_sample.drop(columns=[TARGET])
y_train = train_sample[TARGET]

X_valid = valid_data.drop(columns=[TARGET])
y_valid = valid_data[TARGET]

In [ ]:
X_train = X_train.drop(columns=["date"])
X_valid = X_valid.drop(columns=["date"])

In [ ]:
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_columns = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_columns
        )
    ],
    remainder="passthrough"
)

In [ ]:
def evaluate_model(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    wape = (
        np.abs(y_true - y_pred).sum()
        /
        np.abs(y_true).sum()
    )

    smape = (
        100
        *
        np.mean(
            (
                2*np.abs(y_true-y_pred)
            )
            /
            (
                np.abs(y_true)
                +
                np.abs(y_pred)
                +
                1e-10
            )
        )
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "WAPE": wape,
        "SMAPE": smape
    }

In [ ]:
baseline_prediction = X_valid["lag_1"]

baseline_metrics = evaluate_model(
    y_valid,
    baseline_prediction
)

baseline_metrics

{'MAE': 124.01255151905046,
 'RMSE': np.float64(461.7674651514353),
 'R2': 0.8812808759081001,
 'WAPE': np.float64(0.25663661015004857),
 'SMAPE': np.float64(43.881707325685674)}

In [ ]:
models = {

    "Linear Regression":
        LinearRegression(),

    "Ridge":
        Ridge(),

    "Lasso":
        Lasso(),


    "XGBoost":
        XGBRegressor(
            n_estimators=50,
            learning_rate=0.1,
            max_depth=4,
            random_state=42,
            n_jobs=-1
        ),

    "LightGBM":
        LGBMRegressor(
            n_estimators=50,
            learning_rate=0.1,
            random_state=42,
            verbose=-1
        ),

    "CatBoost":
        CatBoostRegressor(
            iterations=50,
            learning_rate=0.1,
            random_seed=42,
            verbose=0
        )
}

In [ ]:
import time

In [ ]:
print(X_train.shape)
print(X_valid.shape)
print(X_train.memory_usage(deep=True).sum() / 1024**2)

(200000, 40)
(167508, 40)
145.92648220062256


In [ ]:
results = []

trained_models = {}

for model_name, model in models.items():

    print("="*60)
    print(f"Training : {model_name}")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    train_start = time.perf_counter()

    pipeline.fit(
        X_train,
        y_train
    )

    train_time = time.perf_counter() - train_start

    predict_start = time.perf_counter()

    predictions = pipeline.predict(
        X_valid
    )

    prediction_time = time.perf_counter() - predict_start

    metrics = evaluate_model(
        y_valid,
        predictions
    )

    results.append({

        "Model": model_name,

        "MAE": metrics["MAE"],

        "RMSE": metrics["RMSE"],

        "R2": metrics["R2"],

        "WAPE": metrics["WAPE"],

        "SMAPE": metrics["SMAPE"],

        "Train Time": round(train_time,2),

        "Prediction Time": round(prediction_time,2)

    })

    trained_models[model_name] = pipeline

    print("Completed")

Training : Linear Regression
Completed
Training : Ridge
Completed
Training : Lasso
Completed
Training : Random Forest


KeyboardInterrupt: 

In [ ]:
results = pd.DataFrame(results)

results = results.sort_values(
    by="WAPE"
).reset_index(drop=True)

results

In [ ]:
best_model_name = results.iloc[0]["Model"]

best_pipeline = trained_models[
    best_model_name
]

print(best_model_name)

In [ ]:
best_model_name = results.iloc[0]["Model"]

best_pipeline = trained_models[
    best_model_name
]

print(best_model_name)

In [ ]:
import joblib
MODEL_PATH = Path("../artifacts")

MODEL_PATH.mkdir(exist_ok=True)

joblib.dump(
    best_pipeline,
    MODEL_PATH / "best_pipeline.pkl"
)

NameError: name 'best_pipeline' is not defined

In [ ]:
loaded_pipeline = joblib.load(
    MODEL_PATH / "best_pipeline.pkl"
)

loaded_pipeline.predict(
    X_valid.head()
)